In [1]:
import sys
from pathlib import Path
import pandas as pd
import json

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [ ]:
from src.parsing.arelle_parser import load_model_xbrl

html_filepaths = [projectRoot / "data" / "raw" / "fca" /
                  "GSK_PLC" / "GSK_PLC_2024.html"]

# Load the XBRL model
model_xbrl = load_model_xbrl(str(html_filepaths[0]))

In [29]:
from src.parsing.arelle_parser import (debug_fact_attributes,
                                       debug_context_attributes)
debug_fact_attributes(model_xbrl, projectRoot, object_number=50)
debug_context_attributes(model_xbrl, projectRoot, object_number=50)

Attributes written to /workspace/data/processed/debug/fact_attributes_debug.txt (object index 50)
Attributes written to /workspace/data/processed/debug/context_attributes_debug.txt (object index 48)


In [ ]:
from src.parsing.arelle_parser import (safe_dict, extract_context_rows,
                                       extract_fact_rows)

filing_id = "GSK_2024"  # Example filing ID
fact_rows = extract_fact_rows(model_xbrl, filing_id)
print("Fact Rows:", json.dumps(fact_rows[:1], indent=2))

context_rows = extract_context_rows(model_xbrl, filing_id)
print("Context Rows:", json.dumps(safe_dict(context_rows[:1]), indent=2))

Fact Rows: [
  {
    "filing_id": "GSK_2024",
    "context_id": "c-1",
    "raw_name": "ifrs-full:RevenueFromSaleOfGoods",
    "taxonomy_domain": "ifrs-full",
    "data_type": "string",
    "unit_ref": "GBP",
    "decimals": "-6",
    "value_text": "31376000000",
    "value_numeric": null
  }
]
Context Rows: [
  {
    "filing_id": "GSK_2024",
    "context_id": "c-1",
    "entity_scheme": "http://standards.iso.org/iso/17442",
    "entity_identifier": "5493000HZTVUYLO1D793",
    "period_type": "duration",
    "instant": "None",
    "period_start": "2024-01-01",
    "period_end": "2025-01-01",
    "dimensional_qualifier": {}
  }
]


In [48]:
# Convert fact_rows and context_rows to pandas DataFrames
fact_df = pd.DataFrame(fact_rows)
context_df = pd.DataFrame(context_rows)

# Merge the DataFrames on 'context_id'
merged_df = pd.merge(fact_df, context_df, on=['filing_id', 'context_id'], how='inner')

# Display the merged DataFrame
print(len(merged_df))

# Save the merged DataFrame as a CSV file
output_path = (projectRoot / "data" / "processed" / 
               "canonical_facts" / f"{filing_id}.csv")
merged_df.to_csv(output_path, index=False)
print(f"Saved merged DataFrame to {output_path}")

933
Saved merged DataFrame to /workspace/data/processed/canonical_facts/GSK_2024.csv


In [52]:
all_none = merged_df["value_numeric"].isnull().all()
print(f"All values are None: {all_none}")

all_dt_string = (merged_df["data_type"] == "string").all()
print(f"All data types are string: {all_dt_string}")

All values are None: True
All data types are string: True
